# MCP Identity and Non-Human Identity Governance

You are the security engineer for Acme's production support MCP service. Build an evidence-backed identity lifecycle from candidate registration through attested issuance, delegated downstream access, reconciliation, and containment.

**Safety boundary:** everything runs offline with ephemeral training keys and in-memory stores. No cloud, IdP, SPIFFE, Kubernetes, or production credential is contacted.

## Objectives and success criteria

A successful investigation proves that identity is owned and attested, credentials are short-lived and context-bound, delegation only narrows authority, revocation affects acceptance, and inventory drift is measurable.

## 1. Separate identity planes

Human identity, OAuth client identity, workload identity, service instance, immutable artifact, and delegated authority answer different questions. A client name or SPIFFE URI is not authorization by itself. Modern MCP carries client information on each request, so the enforcement point must reconstruct trusted identity and policy each time.

In [ ]:
import json
import runpy
from dataclasses import replace

lab = runpy.run_path('lab.py')
NOW = 1_000
registry, attestor, issuer, record, audit = lab['build_demo'](NOW)
record

### Inspect the governed contract

The record binds stable identity to owner, tenant, purpose, environment, trust domain, runtime selectors, artifact, audiences, scopes, lifetime, lifecycle version, revocation epoch, and review date. Registration consumed an exact two-person approval before activation.

In [ ]:
{
    'identity': record.identity_id,
    'owner': record.owner_id,
    'state': record.state.value,
    'audiences': sorted(record.allowed_audiences),
    'scopes': sorted(record.allowed_scopes),
    'max_ttl_seconds': record.max_ttl_seconds,
    'version': record.version,
}

### Negative test: self-approval

Approval must be performed by an active, authorized principal distinct from the requester.

In [ ]:
try:
    registry.approvals.issue({'scope': 'admin'}, 'security-approver', 'security-approver', NOW)
except lab['GovernanceError'] as exc:
    self_approval_result = exc.code
self_approval_result

## 2. Attest runtime state before issuance

The training attestor signs identity, tenant, trust domain, selectors, immutable artifact digest, observation time, and nonce with Ed25519. Verification requires signature, freshness, replay resistance, and exact governance binding.

In [ ]:
signed_attestation = lab['make_attestation'](attestor, record, NOW + 2, 'notebook-parent')
signed_attestation.evidence.payload()

### Negative test: artifact drift

A valid attestor signature for the wrong artifact is still the wrong workload contract.

In [ ]:
drifted_evidence = replace(signed_attestation.evidence, artifact_digest='b' * 64, nonce='drift')
drifted = attestor.issue(drifted_evidence)
try:
    attestor.verify(drifted, record, NOW + 2)
except lab['GovernanceError'] as exc:
    artifact_drift_result = exc.code
artifact_drift_result

## 3. Issue a short-lived, sender-bound MCP credential

PyJWT signs a real Ed25519 JWT. Issuance intersects the request with the current record and attestation; it never accepts a model-supplied identity as trusted context.

In [ ]:
SENDER = 'sha256:gateway-proof-key-2026'
parent_token = issuer.issue(
    record.identity_id,
    'https://mcp.acme.test/support',
    frozenset({'ticket:read', 'ticket:triage'}),
    record.purpose,
    300,
    SENDER,
    signed_attestation,
    NOW + 2,
    'notebook-issue-1',
)
parent_token.count('.')

The notebook displays only structural facts—not the raw token. Production logs must also omit access tokens, private keys, and attestation signatures.

In [ ]:
parent_claims = issuer.verify(
    parent_token,
    'https://mcp.acme.test/support',
    frozenset({'ticket:read'}),
    SENDER,
    'acme',
    record.purpose,
    NOW + 3,
)
{key: parent_claims[key] for key in ('iss', 'sub', 'aud', 'scope', 'purpose', 'identity_version', 'revocation_epoch')}

### Negative test: cross-service replay

A valid MCP token is not a ticket-API token. Exact audience verification stops cross-service replay.

In [ ]:
try:
    issuer.verify(parent_token, 'https://tickets.acme.test', frozenset(), SENDER, 'acme', record.purpose, NOW + 3)
except lab['GovernanceError'] as exc:
    audience_result = exc.code
audience_result

## 4. Exchange; do not pass through

The MCP server exchanges the parent grant for a child with a different audience, narrower scope, shorter lifetime, fresh sender binding, bounded depth, and parent linkage.

In [ ]:
current_record = registry.get(record.identity_id)
child_attestation = lab['make_attestation'](attestor, current_record, NOW + 4, 'notebook-child')
child_token = issuer.exchange(
    parent_token, 'https://mcp.acme.test/support', 'https://tickets.acme.test',
    frozenset({'ticket:read'}), SENDER, 'sha256:ticket-api-proof-key',
    'acme', record.purpose, 120, child_attestation, NOW + 4, 'notebook-exchange-1',
)
child_claims = issuer.verify(
    child_token, 'https://tickets.acme.test', frozenset({'ticket:read'}),
    'sha256:ticket-api-proof-key', 'acme', record.purpose, NOW + 5,
)
{'scope': child_claims['scope'], 'parent_linked': child_claims['parent_jti'] == parent_claims['jti'], 'depth': child_claims['delegation_depth']}

### Negative test: delegation expansion

Token exchange is only safe when the authorization server's policy enforces attenuation. RFC 8693 defines the exchange, not the least-privilege policy.

In [ ]:
expansion_attestation = lab['make_attestation'](attestor, current_record, NOW + 6, 'notebook-expand')
try:
    issuer.exchange(
        parent_token, 'https://mcp.acme.test/support', 'https://tickets.acme.test',
        frozenset({'ticket:read', 'ticket:delete'}), SENDER, 'sha256:ticket-api-proof-key',
        'acme', record.purpose, 60, expansion_attestation, NOW + 6, 'notebook-expand-1',
    )
except lab['GovernanceError'] as exc:
    expansion_result = exc.code
expansion_result

## 5. Reconcile desired, registered, and observed identity

Runtime evidence can expose a shadow workload and artifact or tenant drift. A missing observation is ambiguous, so the finding starts investigation rather than automatic deletion.

In [ ]:
observation = lab['RuntimeObservation'](
    record.identity_id, 'acme', record.selectors, record.artifact_digest, NOW + 5
)
expected = frozenset({record.identity_id})
findings = lab['reconcile_inventory'](registry, expected, [observation], NOW + 5)
findings

In [ ]:
shadow = lab['RuntimeObservation'](
    'spiffe://acme.test/mcp/server/shadow', 'acme', frozenset({'unknown'}), 'c' * 64, NOW + 5
)
drift = replace(observation, tenant_id='other', artifact_digest='b' * 64)
drift_findings = lab['reconcile_inventory'](registry, expected, [drift, shadow], NOW + 5)
[(finding.code, finding.severity) for finding in drift_findings]

## 6. Measure coverage with explicit denominators

Inventory coverage and runtime governance coverage are different. Keep expected identities and observed identities as separate populations.

In [ ]:
metrics = lab['governance_metrics'](registry, expected, [observation], findings)
metrics

## 7. Contain the workload and prove old-token rejection

Suspension changes the current registry version and lifecycle state. The verifier checks that live state rather than trusting a token until expiry.

In [ ]:
before_containment = registry.get(record.identity_id)
registry.suspend(record.identity_id, before_containment.version, NOW + 10, 'suspected workload compromise')
try:
    issuer.verify(parent_token, 'https://mcp.acme.test/support', frozenset(), SENDER, 'acme', record.purpose, NOW + 11)
except lab['GovernanceError'] as exc:
    containment_result = exc.code
containment_result

## 8. Inspect evidence without secrets

Events retain identity, tenant, decision, stable reason, policy version, time, and correlation. The raw JWT and attestation signature are absent.

In [ ]:
[
    {
        'type': event.event_type, 'identity': event.identity_id,
        'decision': event.decision, 'reason': event.reason,
        'correlation': event.correlation_id,
    }
    for event in audit.events
]

## Production gap analysis

This notebook proves application invariants with real signed JWTs. It does **not** prove platform attestation, SPIFFE federation, TLS or DPoP, HSM protection, Kubernetes admission, distributed consistency, cache invalidation, IdP availability, or end-to-end revocation convergence. Map each production claim to its real enforcement point, test, telemetry, and owner.

## Reflection

1. Which identity planes would your current MCP logs conflate?
2. What runtime facts can your platform attest without trusting workload-supplied labels?
3. How quickly can every accepting resource reject a compromised identity, and how is that measured?
4. Which owner-departure feed is authoritative, and what happens when it is delayed?
5. Design a chaos test for a stale verifier cache during emergency revocation.
6. Explain why a short-lived bearer token can still be replayed before expiry.

## Next steps

Run `uv run pytest -q tests/test_course_24_nonhuman_identity.py`, then replace one fixture at a time with your platform's identity, registry, policy, and evidence adapters. Preserve the negative tests and add an independent protected-resource outcome oracle.